In [4]:
import os
import json


In [5]:

base_folder = "/Users/mateo/pyronear/vision/smoke-localization/site_data/laluque/captured_poses/192_168_1_11"
shift_json_path = base_folder + "_angle_shift.json"
ref_json_path = base_folder + "_ref_azimuth.json"
output_path = base_folder + "_full_azimuth.json"

In [6]:

# ==== LOAD DATA ====

with open(shift_json_path, 'r') as f:
    angle_shifts = json.load(f)

with open(ref_json_path, 'r') as f:
    ref_azimuths = json.load(f)

# Get all unique images
all_images = sorted(set(
    [key.split("__")[0] for key in angle_shifts] +
    [key.split("__")[1] for key in angle_shifts] +
    list(ref_azimuths.keys())
))

# Build image index mappings
image_to_index = {name: idx for idx, name in enumerate(all_images)}
index_to_image = {idx: name for name, idx in image_to_index.items()}

# Initialize azimuth map with known values
azimuth_map = {}
for img, values in ref_azimuths.items():
    azimuth_map[img] = values["computed_center_azimuth"]

# ==== FORWARD PROPAGATION (LEFT → RIGHT) ====

for i in range(len(all_images) - 1):
    img1 = all_images[i]
    img2 = all_images[i + 1]
    key = f"{img1}__{img2}"

    if key in angle_shifts:
        shift = angle_shifts[key]["angle_deg"]
        if img1 in azimuth_map and img2 not in azimuth_map:
            azimuth_map[img2] = (azimuth_map[img1] + shift) % 360

# ==== BACKWARD PROPAGATION (RIGHT → LEFT) ====

for i in reversed(range(1, len(all_images))):
    img1 = all_images[i - 1]
    img2 = all_images[i]
    key = f"{img1}__{img2}"

    if key in angle_shifts:
        shift = angle_shifts[key]["angle_deg"]
        if img2 in azimuth_map and img1 not in azimuth_map:
            azimuth_map[img1] = (azimuth_map[img2] - shift) % 360

# ==== SAVE RESULTS ====

with open(output_path, 'w') as f:
    json.dump(azimuth_map, f, indent=2)

print(f"✅ Computed azimuths for {len(azimuth_map)} images")
print(f"➡️  Saved to {output_path}")

✅ Computed azimuths for 6 images
➡️  Saved to /Users/mateo/pyronear/vision/smoke-localization/site_data/laluque/captured_poses/192_168_1_11_full_azimuth.json
